# 🔥 YOLOvX — SAM2 Inference Server
> Auto-generated by your annotation platform. Just run all cells!

**Requirements:** GPU runtime (T4 recommended)
Runtime → Change runtime type → T4 GPU → Save

In [ ]:
# ⚡ CELL 1 — Install & Setup (Run Once per Session Start)
!pip install -q fastapi uvicorn python-multipart requests opencv-python-headless git+https://github.com/facebookresearch/segment-anything-2.git

import os

# Download model weights (skipped if cached)
if not os.path.exists('sam2_hiera_large.pt'):
    os.system('wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt')
    print('✅ Model downloaded.')
else:
    print('✅ Model cached, skipping download.')

# Download cloudflared tunnel binary
if not os.path.exists('cloudflared-linux-amd64'):
    os.system('wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64')
    os.system('chmod +x cloudflared-linux-amd64')
    print('✅ Cloudflared ready.')
else:
    print('✅ Cloudflared cached.')

# Write server.py
SUPABASE_URL = "https://base.wiserly.org/"

server_code = '''
import os, cv2, numpy as np, requests
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
from pydantic import BaseModel
from PIL import Image
from io import BytesIO

SUPABASE_URL = "''' + SUPABASE_URL + '''"

app = FastAPI()
app.add_middleware(__import__("fastapi.middleware.cors", fromlist=["CORSMiddleware"]).CORSMiddleware,
    allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

sam2_model = build_sam2("sam2_hiera_l.yaml", "sam2_hiera_large.pt", device="cuda")
predictor = SAM2ImagePredictor(sam2_model)
current_img_id = None
current_img_shape = None

class Req(BaseModel):
    image_path: str
    click_x: float
    click_y: float

@app.get("/health")
def health(): return {"status": "ok"}

@app.post("/segment")
async def segment(req: Req):
    global current_img_id, current_img_shape
    full_url = req.image_path if "http" in req.image_path else f"{SUPABASE_URL.rstrip(\'/')}/storage/v1/object/public/datasets/{req.image_path}"
    if current_img_id != req.image_path:
        try:
            data = requests.get(full_url).content
            img = np.array(Image.open(BytesIO(data)).convert("RGB"))
            current_img_shape = img.shape
            predictor.set_image(img)
            current_img_id = req.image_path
        except Exception as e:
            return {"error": str(e)}
    orig_h, orig_w = current_img_shape[:2]
    cx = float(np.clip(req.click_x, 0, orig_w - 1))
    cy = float(np.clip(req.click_y, 0, orig_h - 1))
    masks, scores, _ = predictor.predict(point_coords=np.array([[cx, cy]]), point_labels=np.array([1]), multimask_output=True)
    best = masks[np.argmax(scores)]
    m8 = best.astype(np.uint8) * 255
    sm = cv2.GaussianBlur(m8, (7,7), 0)
    _, bn = cv2.threshold(sm, 127, 255, cv2.THRESH_BINARY)
    cs, _ = cv2.findContours(bn, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs: return {"polygon": [], "image_size": [orig_w, orig_h]}
    lg = max(cs, key=cv2.contourArea)
    eps = 0.005 * cv2.arcLength(lg, True)
    poly = cv2.approxPolyDP(lg, eps, True)
    return {"polygon": poly.flatten().tolist(), "image_size": [orig_w, orig_h]}
'''

with open('server.py', 'w') as f:
    f.write(server_code)
print('\n✅ Setup complete! Run Cell 2 to start inference.')

## 🚀 Cell 2 — Start Server (Run each session)
Run this every time you open the notebook. Takes ~30 seconds.

In [ ]:
# 🚀 CELL 2 — Start Inference Server (Run Each Session)
import threading, uvicorn, subprocess, time, re, sys
sys.path.insert(0, '.')

TUNNEL_URL = None

def start_tunnel():
    global TUNNEL_URL
    p = subprocess.Popen(['./cloudflared-linux-amd64', 'tunnel', '--url', 'http://localhost:8000'],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    time.sleep(3)
    for line in iter(p.stderr.readline, b''):
        line = line.decode('utf-8')
        if '.trycloudflare.com' in line:
            m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
            if m:
                TUNNEL_URL = m.group(0)
                print(f'\n\n🔥 COPY THIS URL INTO YOUR APP:\n\n  {TUNNEL_URL}\n\n')
                break

def run_server():
    import server
    uvicorn.run(server.app, host='0.0.0.0', port=8000, log_level='warning')

print('🚀 Starting SAM2 server...')
threading.Thread(target=start_tunnel, daemon=True).start()
threading.Thread(target=run_server, daemon=True).start()

try:
    while True: time.sleep(1)
except KeyboardInterrupt:
    print('Stopped.')